# 📊 Cross Validation Analysis
Stratified K-Fold Cross Validation

Run AFTER `02_training_testing.ipynb`

---
### Cells
1. Imports & Config
2. Load Image Paths
3. Image Loader
4. CV Model Builder
5. Run K-Fold CV
6. Results & Plots

## Cell 1 — Imports & Config

In [1]:
import os, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')
from pathlib import Path
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, classification_report
from sklearn.utils.class_weight import compute_class_weight
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import EfficientNetB4
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout, BatchNormalization
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from PIL import Image
import gc

DATASET_PATH = r'D:\Files\AI X-Ray Detection\Dataset_Cleaned'
ORIG_PATH    = r'D:\Files\AI X-Ray Detection\Dataset'
RESULTS_PATH = r'D:\Files\AI X-Ray Detection\Reports'
CLASSES      = ['normal', 'pneumonia', 'tuberculosis']
IMG_SIZE     = (224, 224)
N_FOLDS      = 5
EPOCHS_CV    = 10
BATCH_SIZE   = 16

if not os.path.exists(DATASET_PATH):
    DATASET_PATH = ORIG_PATH

os.makedirs(RESULTS_PATH, exist_ok=True)
print(f'✅ Config ready — {N_FOLDS}-Fold CV with {EPOCHS_CV} epochs each')

✅ Config ready — 5-Fold CV with 10 epochs each


## Cell 2 — Load Image Paths

In [2]:
all_paths  = []
all_labels = []
class_to_idx = {cls: idx for idx, cls in enumerate(CLASSES)}
train_path   = os.path.join(DATASET_PATH, 'train')

for cls in CLASSES:
    cls_path = os.path.join(train_path, cls)
    if not os.path.isdir(cls_path):
        print(f'  ⚠️  Missing: {cls_path}')
        continue
    files = [f for f in os.listdir(cls_path)
             if Path(f).suffix.lower() in {'.jpg','.jpeg','.png'}]
    for fname in files:
        all_paths.append(os.path.join(cls_path, fname))
        all_labels.append(class_to_idx[cls])

all_paths  = np.array(all_paths)
all_labels = np.array(all_labels)
print(f'Total training images: {len(all_paths)}')
for cls, idx in class_to_idx.items():
    print(f'  {cls:<18}: {np.sum(all_labels==idx)}')

  ⚠️  Missing: D:\Files\AI X-Ray Detection\Dataset\train\normal
  ⚠️  Missing: D:\Files\AI X-Ray Detection\Dataset\train\pneumonia
  ⚠️  Missing: D:\Files\AI X-Ray Detection\Dataset\train\tuberculosis
Total training images: 0
  normal            : 0
  pneumonia         : 0
  tuberculosis      : 0


## Cell 3 — Image Loader

In [3]:
def load_images_batch(paths, img_size=(224, 224)):
    imgs = []
    for p in paths:
        try:
            img = Image.open(p).convert('RGB').resize(img_size)
            imgs.append(np.array(img) / 255.0)
        except:
            imgs.append(np.zeros((*img_size, 3)))
    return np.array(imgs, dtype=np.float32)

print('✅ Image loader ready')

✅ Image loader ready


## Cell 4 — CV Model Builder

In [4]:
def build_cv_model(num_classes):
    base = EfficientNetB4(weights='imagenet', include_top=False, input_shape=(224,224,3))
    for layer in base.layers:
        layer.trainable = False
    for layer in base.layers[-20:]:
        layer.trainable = True
    x   = base.output
    x   = GlobalAveragePooling2D()(x)
    x   = BatchNormalization()(x)
    x   = Dense(256, activation='relu')(x)
    x   = Dropout(0.5)(x)
    out = Dense(num_classes, activation='softmax')(x)
    model = Model(inputs=base.input, outputs=out)
    model.compile(optimizer=Adam(learning_rate=0.0001),
                  loss='categorical_crossentropy', metrics=['accuracy'])
    return model

print('✅ CV model builder ready')

✅ CV model builder ready


## Cell 5 — Run K-Fold Cross Validation

In [1]:
skf          = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
fold_results = []
aug          = ImageDataGenerator(rotation_range=10, horizontal_flip=True,
                                   width_shift_range=0.08, height_shift_range=0.08)

for fold_idx, (train_idx, val_idx) in enumerate(skf.split(all_paths, all_labels)):
    print(f'\n── Fold {fold_idx+1}/{N_FOLDS} ──')

    train_paths  = all_paths[train_idx]
    val_paths    = all_paths[val_idx]
    train_labels = all_labels[train_idx]
    val_labels   = all_labels[val_idx]
    print(f'  Train: {len(train_paths)} | Val: {len(val_paths)}')

    cw      = compute_class_weight('balanced', classes=np.unique(train_labels), y=train_labels)
    cw_dict = dict(enumerate(cw))
    model   = build_cv_model(len(CLASSES))

    y_train_oh = tf.keras.utils.to_categorical(train_labels, len(CLASSES))
    y_val_oh   = tf.keras.utils.to_categorical(val_labels,   len(CLASSES))

    print('  Loading images...')
    X_train = load_images_batch(train_paths)
    X_val   = load_images_batch(val_paths)

    callbacks = [
        EarlyStopping(monitor='val_accuracy', patience=4, restore_best_weights=True, verbose=0),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=1e-7, verbose=0)
    ]

    history = model.fit(
        aug.flow(X_train, y_train_oh, batch_size=BATCH_SIZE),
        epochs=EPOCHS_CV, validation_data=(X_val, y_val_oh),
        class_weight=cw_dict, callbacks=callbacks, verbose=1
    )

    y_pred    = np.argmax(model.predict(X_val, verbose=0), axis=1)
    fold_acc  = float(accuracy_score(val_labels, y_pred))
    best_val  = max(history.history['val_accuracy'])
    rpt       = classification_report(val_labels, y_pred, target_names=CLASSES, output_dict=True)

    print(f'  Fold {fold_idx+1} Accuracy: {fold_acc*100:.2f}%  Best Val: {best_val*100:.2f}%')

    row = {'fold': fold_idx+1, 'accuracy': round(fold_acc,4), 'best_val': round(float(best_val),4)}
    for cls in CLASSES:
        if cls in rpt:
            row[f'{cls}_precision'] = round(rpt[cls]['precision'], 4)
            row[f'{cls}_recall']    = round(rpt[cls]['recall'], 4)
            row[f'{cls}_f1']        = round(rpt[cls]['f1-score'], 4)
    fold_results.append(row)

    del model, X_train, X_val; gc.collect()

print('\n✅ Cross validation complete')

NameError: name 'StratifiedKFold' is not defined

## Cell 6 — Results & Plots

In [ ]:
df_cv    = pd.DataFrame(fold_results)
accs     = df_cv['accuracy'].values
mean_acc = float(np.mean(accs))
std_acc  = float(np.std(accs))

print(f'\n{"="*55}')
print(f'  {N_FOLDS}-FOLD CROSS VALIDATION RESULTS')
print(f'{"="*55}')
print(f'  {"Fold":<6} {"Accuracy":>12} {"Val Acc":>10}')
print(f'  {"-"*30}')
for _, row in df_cv.iterrows():
    print(f'  {int(row["fold"]):<6} {row["accuracy"]:>12.4f} {row["best_val"]:>10.4f}')

print(f'\n  Mean : {mean_acc:.4f} ± {std_acc:.4f}')
print(f'  Min  : {np.min(accs):.4f}')
print(f'  Max  : {np.max(accs):.4f}')

print(f'\n  Per-class Recall Across Folds:')
for cls in CLASSES:
    col = f'{cls}_recall'
    if col in df_cv.columns:
        vals = df_cv[col].values
        print(f'    {cls:<18} mean={np.mean(vals):.4f} ± {np.std(vals):.4f}')

df_cv.to_csv(os.path.join(RESULTS_PATH,'cross_validation_results.csv'), index=False)

# Plots
folds  = df_cv['fold'].values
colors = ['#4f86f7','#ef4444','#a855f7']

fig, axes = plt.subplots(1, 3, figsize=(17, 5))
fig.suptitle(f'Stratified {N_FOLDS}-Fold Cross Validation', fontsize=13, fontweight='bold')

axes[0].plot(folds, df_cv['accuracy'], 'o-', color='#4f86f7', linewidth=2, label='Test Acc')
axes[0].plot(folds, df_cv['best_val'], 's-', color='#a855f7', linewidth=2, label='Val Acc')
axes[0].axhline(y=mean_acc, color='green', linestyle='--', label=f'Mean={mean_acc:.3f}')
axes[0].fill_between(folds, mean_acc-std_acc, mean_acc+std_acc, alpha=0.15, color='green')
axes[0].set_title('Accuracy per Fold'); axes[0].set_xlabel('Fold'); axes[0].set_ylabel('Accuracy')
axes[0].legend(fontsize=8); axes[0].grid(True, alpha=0.3); axes[0].set_xticks(folds)

for cls, color in zip(CLASSES, colors):
    col = f'{cls}_recall'
    if col in df_cv.columns:
        axes[1].plot(folds, df_cv[col], 'o-', color=color, linewidth=2, label=cls)
axes[1].axhline(y=0.8, color='orange', linestyle='--', label='0.80 target')
axes[1].set_title('Per-class Recall per Fold'); axes[1].set_xlabel('Fold')
axes[1].legend(fontsize=8); axes[1].grid(True, alpha=0.3); axes[1].set_xticks(folds)

axes[2].boxplot(accs, patch_artist=True,
    boxprops=dict(facecolor='#4f86f7', alpha=0.5),
    medianprops=dict(color='#a855f7', linewidth=2))
for i, acc in enumerate(accs):
    axes[2].scatter(1, acc, color='#4f86f7', zorder=5, s=80)
    axes[2].text(1.08, acc, f'F{i+1}: {acc:.3f}', va='center', fontsize=9)
axes[2].set_title(f'Accuracy Distribution\nμ={mean_acc:.3f} ± {std_acc:.3f}')
axes[2].set_ylabel('Accuracy'); axes[2].set_xticks([]); axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_PATH,'cross_validation_analysis.png'), dpi=150)
plt.show()

stability = 'Stable ✅' if std_acc < 0.05 else 'High variance ⚠️'
print(f'\n  Stability: {stability}')
print(f'  Results saved → {RESULTS_PATH}')